# A2A on AgentCore: From localhost to AWS

In Notebook 1 your Refund agent ran on `127.0.0.1:9000`. Great for a demo. Useless to a peer in another VPC, another account, or another company.

This notebook ships that same agent to **Amazon Bedrock AgentCore Runtime**, so any authorized A2A client anywhere can discover and call it.

**The one idea:**

> AgentCore Runtime is a **transparent A2A proxy**. It terminates auth, injects a session, and passes your JSON-RPC through **unmodified**.

Because the proxy is transparent, your server code barely changes. The agent logic does not change at all. You add three small production touches and deploy.


## What changes from Notebook 1

Almost nothing. AgentCore expects a container listening on **port 9000 at root `/`**, which is already the Strands default. You add exactly three things.

| Addition | Why |
|---|---|
| Read `http_url` from `AGENTCORE_RUNTIME_URL` | So the Agent Card advertises the real public URL, not localhost |
| A `GET /ping` returning `{"status":"healthy"}` | AgentCore health checks the container |
| Mount the A2A app at root and set `serve_at_root=True` | Handles load-balancer path prefixes cleanly |

### The port contract

AgentCore routes by protocol. Match the port and path or the container never registers.

| Protocol | Port | Path |
|---|---|---|
| HTTP (default agents) | 8080 | `/invocations` |
| MCP | 8000 | `/mcp` |
| **A2A** | **9000** | **`/`** |

Strands' A2A defaults already hit the A2A row. That is not luck. It is why Strands plus AgentCore is a short walk.


---
## Part A: Make the server AgentCore-ready

Same Refund agent. Three additions, wrapped in a FastAPI app so we can add the `/ping` route alongside the mounted A2A app.

This whole part runs **locally with no AWS credentials**. The deployed artifact is byte-for-byte the same file you boot here.

In [ ]:
%%writefile my_a2a_server.py
import os
from strands import Agent, tool
from strands.multiagent.a2a import A2AServer
from strands.models import BedrockModel
from fastapi import FastAPI
import uvicorn

@tool
def look_up_booking(pnr: str) -> dict:
    """Look up an airline booking by PNR. Returns fare and refund eligibility."""
    return {"pnr": pnr, "fare_usd": 420.0, "refundable": True, "penalty_usd": 50.0}

@tool
def process_refund(pnr: str, amount_usd: float) -> dict:
    """Process a refund for a booking. Returns a confirmation id."""
    return {"pnr": pnr, "refunded_usd": amount_usd, "confirmation": "RF-" + pnr}

# Addition 1: advertise the real URL when AgentCore sets it; fall back to localhost.
runtime_url = os.environ.get("AGENTCORE_RUNTIME_URL", "http://127.0.0.1:9000/")

agent = Agent(
    name="Refund Agent",
    description="Handles airline refund eligibility and processing.",
    model=BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0"),
    tools=[look_up_booking, process_refund],
    callback_handler=None,
)

a2a_server = A2AServer(agent=agent, http_url=runtime_url, serve_at_root=True)

app = FastAPI()

# Addition 2: health check for the AgentCore container.
@app.get("/ping")
def ping():
    return {"status": "healthy"}

# Addition 3: mount the A2A app at root.
app.mount("/", a2a_server.to_fastapi_app())

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=9000)

In [ ]:
%%writefile requirements.txt
strands-agents[a2a]
bedrock-agentcore
strands-agents-tools

Boot it locally and prove two things: the health check answers, and the Agent Card serves at root with both skills. This is exactly what AgentCore will probe after deploy.

In [ ]:
import subprocess, sys, time, urllib.request, json

SERVER_URL = "http://127.0.0.1:9000"

def launch(script, ready_url, timeout=90):
    proc = subprocess.Popen([sys.executable, script])
    deadline = time.time() + timeout
    while time.time() < deadline:
        if proc.poll() is not None:
            raise RuntimeError(f"server exited early: {proc.returncode}")
        try:
            with urllib.request.urlopen(ready_url, timeout=2) as r:
                if r.status == 200:
                    print("ready ->", ready_url); return proc
        except Exception:
            time.sleep(1)
    proc.terminate(); raise RuntimeError("not ready in time")

server_proc = launch("my_a2a_server.py", SERVER_URL + "/ping")

with urllib.request.urlopen(SERVER_URL + "/ping") as r:
    print("ping:", r.read().decode())

with urllib.request.urlopen(SERVER_URL + "/.well-known/agent-card.json") as r:
    card = json.load(r)
print("card url:   ", card["url"])
print("protocol:   ", card["protocolVersion"])
print("skills:     ", [s["name"] for s in card["skills"]])

server_proc.terminate()
try: server_proc.wait(timeout=10)
except Exception: server_proc.kill()
print("local server stopped")

---
## Part B: Deploy to AgentCore

The starter toolkit turns the file above into a deployed runtime with two commands.

From this point you need AWS credentials for account `452203592848`, region `us-east-1`. These are **terminal commands**, not notebook cells.

```bash
pip install bedrock-agentcore-starter-toolkit

# point the toolkit at your server file and declare the protocol
agentcore configure -e my_a2a_server.py --protocol A2A

# build, push, and deploy; an ARN comes back
agentcore deploy
# returns an ARN like:
# arn:aws:bedrock-agentcore:us-east-1:452203592848:runtime/my_a2a_server-xyz123
```

`--protocol A2A` is the line that matters. It tells AgentCore to expect the A2A port-and-path contract (9000 at root) and to run as a transparent JSON-RPC proxy.

### Newer path: the AgentCore CLI

AWS now ships a dedicated CLI that is becoming the recommended tooling. Same idea, with hot reload and broader framework support.

```bash
npm install -g @aws/agentcore
```

Either path produces the same thing: a runtime ARN and a managed HTTPS endpoint.

---
## Part C: Auth and sessions

Localhost had no auth. AgentCore does not give you that option, and that is the point.

| Concern | What AgentCore does |
|---|---|
| **Who is calling** | OAuth 2.0 via a Cognito user pool **or** IAM SigV4 |
| **Keeping callers separate** | Injects a per-session header `X-Amzn-Bedrock-AgentCore-Runtime-Session-Id` |

The session header is the multi-tenant safety net. Two passengers hitting TravelMind at the same time get two isolated sessions. Their state never bleeds into each other, and you wrote zero code for it.


---
## Part D: Invoke the deployed agent

Discovery over AgentCore is the same GET, now against the managed HTTPS URL and with an auth token. The card comes back identical to localhost, which is the proxy being transparent.

Fill in your runtime URL and bearer token (from Cognito) and run.

In [ ]:
import httpx

# From `agentcore deploy` output and your Cognito token.
RUNTIME_URL = "https://<your-agentcore-runtime-url>"   # replace
BEARER_TOKEN = "<cognito-access-token>"                 # replace

headers = {"Authorization": f"Bearer {BEARER_TOKEN}"}

# 1) Discover: fetch the card through the proxy.
card = httpx.get(RUNTIME_URL + "/.well-known/agent-card.json", headers=headers, timeout=30).json()
print("discovered:", card["name"], "| skills:", [s["name"] for s in card["skills"]])

For full A2A calls, point the Strands client tools at the runtime URL and pass the auth header through `httpx_client_args`. Now an orchestrator can discover and call the deployed peer exactly as it did on localhost.

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands_tools.a2a_client import A2AClientToolProvider

provider = A2AClientToolProvider(
    known_agent_urls=[RUNTIME_URL],
    httpx_client_args={"headers": {"Authorization": f"Bearer {BEARER_TOKEN}"}},
)

orchestrator = Agent(
    name="TravelMind",
    model=BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0"),
    tools=provider.tools,
    callback_handler=None,
)

reply = orchestrator(
    "Discover the deployed agent and ask whether PNR JX48Q2 is refundable and for how much."
)
print(reply.message)

---
## Part E: Reading errors in production

AgentCore returns errors as JSON-RPC. The trap: the **HTTP status is 200** even when the call failed. The real signal is `error.code` inside the JSON body. Read the body, not the status line.

| `error.code` | Meaning | Maps to | What to do |
|---|---|---|---|
| -32501 | ResourceNotFound | 404 | Wrong runtime URL or ARN |
| -32502 | Validation | 400 | Malformed JSON-RPC or bad params |
| -32503 | Throttling | 429 | Back off and retry |
| -32504 | Conflict | 409 | Duplicate or conflicting session |
| **-32505** | **RuntimeClientError** | **424** | **Your agent crashed. Check CloudWatch logs** |
| (AccessDenied) | Auth failed | 403 | Token expired or missing scope |

`-32505` is the one you will meet most. It means the proxy reached your agent and your agent threw. A `424` with a healthy `/ping` means the bug is in your code, not the wiring. Go straight to CloudWatch.


In [ ]:
def read_a2a_error(response_json: dict):
    """A2A errors ride inside a 200 response. Inspect the body, not the status code."""
    err = response_json.get("error")
    if not err:
        return None
    hint = {
        -32501: "ResourceNotFound: check runtime URL / ARN",
        -32502: "Validation: malformed request or params",
        -32503: "Throttling: back off and retry",
        -32504: "Conflict: duplicate/conflicting session",
        -32505: "RuntimeClientError: agent crashed -> check CloudWatch",
    }.get(err.get("code"), "see error.message")
    return {"code": err.get("code"), "message": err.get("message"), "hint": hint}

# Example of a crashed-agent response shape:
print(read_a2a_error({"jsonrpc": "2.0", "id": 1,
                      "error": {"code": -32505, "message": "runtime client error"}}))

---
## Part F: The payoff, cross-framework

Here is what AgentCore plus A2A unlocks, and why it is more than microservices with JSON.

AWS's own reference architecture for incident response runs three agents built in **three different frameworks**, all speaking A2A on AgentCore:

```mermaid
flowchart LR
    H["Host / orchestrator<br/>Google ADK"] -->|A2A| M["Monitoring agent<br/>Strands"]
    H -->|A2A| O["Operations agent<br/>OpenAI Agents SDK"]
    M -.discovers via card.-> H
    O -.discovers via card.-> H
```

The host was written by one team in Google ADK. The monitoring agent by another in Strands. The operations agent by a third in the OpenAI Agents SDK. None imports the others. They cooperate because they share one contract: A2A.

> That is the line between A2A and plain microservices. Microservices share an internal API you designed. A2A agents share a **public discovery and message standard**, so a vendor's agent and yours interoperate with no glue code and no prior agreement.


## Recap

- AgentCore Runtime is a transparent A2A proxy. Your server code gains three small touches and nothing else.
- Match the A2A contract: **port 9000, root `/`**. Strands defaults already do.
- Deploy with `agentcore configure --protocol A2A` then `agentcore deploy`. An ARN comes back. The newer `@aws/agentcore` CLI is the same idea.
- Auth is OAuth 2.0 (Cognito) or IAM SigV4. Sessions are isolated by an injected header you never manage.
- Errors arrive as JSON-RPC inside a `200`. Read `error.code`. `-32505` means your agent crashed; check CloudWatch.
- The endgame: agents from different teams and different frameworks cooperate over one standard, with no shared code.

You took a localhost demo to a deployed, authenticated, multi-tenant agent that any A2A client on earth can reach. That is the whole arc, from scratch to production.
